In [1]:
import os
import json
import math
import torch
import random
import multiprocessing as mp
from torch import nn
from tqdm import tqdm
from array import array
from pathlib import Path
from torch.nn import functional as F
from collections import Counter, defaultdict
from torch.utils.data import DataLoader
from torch.utils.data import IterableDataset, get_worker_info

In [2]:
def _read_chunk_by_chunk_from_offset(file_path, end_token, offset=0, chunk_size=8 * 1024 * 1024):
    end_token_bytes = end_token.encode('utf-8')
    pos = offset
    leftover = b''

    with open(file_path, 'rb') as f:
        f.seek(offset)
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            data = leftover + chunk
            parts = data.split(end_token_bytes)
            leftover = parts.pop()
            for p in parts:
                if p:
                    pos += len(p) + len(end_token_bytes)
                    yield p.decode('utf-8') + end_token, pos
        if leftover:
            pos += len(leftover)
            yield leftover.decode('utf-8'), pos

In [3]:
class ProteinDataset(IterableDataset):
    def __init__(self, corpus_path, tokenizer,
                  context_length,end_token="<|endofprotein|>", buffer_size=500,
                  resume_state=None, position_array=None, idx_array=None):
        super().__init__()
        self.corpus_path = corpus_path
        self.tokenizer = tokenizer
        self.context_length = context_length
        self.end_token = end_token
        self.buffer_size = buffer_size
        self.resume_state = resume_state or {}
        self.position_array = position_array
        self.idx_array = idx_array

    def __iter__(self):
        token_buffer = []
        shuffle_buffer = []
        worker_info = get_worker_info()
        num_workers = worker_info.num_workers if worker_info else 1
        worker_id = worker_info.id if worker_info else 0

        byte_offset, start_idx = self.resume_state.get(worker_id, (0, 0))
        idx = start_idx
        for protein, byte_pos in _read_chunk_by_chunk_from_offset(self.corpus_path, self.end_token, byte_offset):
            current_idx = idx
            idx += 1
            
            if current_idx % num_workers != worker_id:
                continue
            
            tokens = self.tokenizer.encode(protein, allowed_special={self.end_token})
            token_buffer.extend(tokens)

            while len(token_buffer) >= self.context_length + 1:
                chunk = token_buffer[:self.context_length + 1]
                input_ids = chunk[:-1]
                target_ids = chunk[1:]

                sample = (
                    torch.tensor(input_ids, dtype=torch.long),
                    torch.tensor(target_ids, dtype=torch.long),
                    byte_pos,
                    idx,
                )

                shuffle_buffer.append(sample)
                if len(shuffle_buffer) >= self.buffer_size:
                    random_idx = random.randrange(len(shuffle_buffer))
                    yielded = shuffle_buffer.pop(random_idx)
                    if self.position_array is not None:
                        self.position_array[worker_id] = yielded[2]
                        self.idx_array[worker_id] = yielded[3]
                    yield yielded[0], yielded[1]

                token_buffer = token_buffer[self.context_length:]

        while shuffle_buffer:
            random_idx = random.randrange(len(shuffle_buffer))
            yielded = shuffle_buffer.pop(random_idx)
            if self.position_array is not None:
                self.position_array[worker_id] = yielded[2]
                self.idx_array[worker_id] = yielded[3]
            yield yielded[0], yielded[1]

In [4]:
def _iter_words_from_file(file_path, end_token, chunk_size=8 * 1024 * 1024):
    buffer=''
    with open(file_path, 'r', encoding='utf-8') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            buffer += chunk
            parts = buffer.split(end_token)
            buffer = parts.pop()
            for p in parts:
                if p:
                    yield p
        if buffer:
            yield buffer

In [5]:
PAIR_BASE = 1 << 20

def _pack(a, b):
    return a * PAIR_BASE + b

def _unpack(p):
    return divmod(p, PAIR_BASE)


In [6]:
class BPETokenizer:
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
        self.bpe_merges = {}
        self.merge_ranks = {}
        self.unk_token = '<|unk|>'
        self.end_token = '<|endofprotein|>'

    def _initialize_vocab(self, unique_chars, special_tokens):
        self.vocab = {
            token_id: token
            for token_id, token in enumerate(unique_chars)
        }

        self.inverse_vocab = {
            token: token_id
            for token_id, token in self.vocab.items()
        }

        for token in sorted(special_tokens):
            if token not in self.inverse_vocab:
                token_id = len(self.vocab)

                self.vocab[token_id] = token
                self.inverse_vocab[token] = token_id

    @staticmethod
    def _convert_to_tuple(value):
        if isinstance(value, list):
            return tuple(
                BPETokenizer._convert_to_tuple(item)
                for item in value
            )
        return value

    @staticmethod
    def _reservoir_sample(file_path, end_token, sample_size, seed=0):
        rng = random.Random(seed)

        reservoir = []
        seen = 0

        for protein in _iter_words_from_file(file_path, end_token):
            if not protein:
                continue
            seen += 1
            if len(reservoir) < sample_size:
                reservoir.append(protein)
                continue
            j = rng.randrange(seen)
            if j < sample_size:
                reservoir[j] = protein
        return reservoir

    @staticmethod
    def _collect_unique_chars(file_path, end_token):
        unique_chars = set()
        for protein in _iter_words_from_file(file_path, end_token):
            unique_chars.update(protein)
        return unique_chars

    def _encode_word_to_array(self, word):
        unk_id = self.inverse_vocab[self.unk_token]
        return array(
            "I",
            (
                self.inverse_vocab.get(char, unk_id)
                for char in word
            )
        )

    def _build_training_words(self, file_path, 
                              end_token, sample_size, seed):
        if sample_size is not None:
            proteins = self._reservoir_sample(
                file_path,
                end_token,
                sample_size,
                seed
            )
            return [
                self._encode_word_to_array(protein)
                for protein in proteins
            ]
        words = []
        for protein in _iter_words_from_file(file_path, end_token):
            if protein:
                words.append(
                    self._encode_word_to_array(protein)
                )
        return words

    @staticmethod
    def _build_pair_statistics(words):
        pair_counts = Counter()
        pair_to_words = defaultdict(set)
        for word_index, word in enumerate(words):
            for i in range(len(word)-1):
                pair = _pack(word[i], word[i+1])
                pair_counts[pair] += 1
                pair_to_words[pair].add(word_index)
        return pair_counts, pair_to_words

    @staticmethod
    def _merge_word(word, best_a, best_b, new_id):
        merged = array("I")
        i = 0
        while i < len(word):
            if(i < len(word) - 1 and word[i] == best_a and word[i+1] == best_b):
                merged.append(new_id)
                i+=2
            else:
                merged.append(word[i])
                i+=1
        return merged

    def train(self, file_path, vocab_size,
               allowed_special=None, sample_size=None, seed=0):
        if allowed_special is None:
            allowed_special = { self.end_token }

        special_tokens = set(allowed_special)
        special_tokens.add(self.unk_token)
        end_token = self.end_token

        if vocab_size <= 0:
            raise ValueError(
                "vocab_size must be greater than 0"
            )

        if vocab_size >= PAIR_BASE:
            raise ValueError(
                f"vocab_size must be less than PAIR_BASE ({PAIR_BASE})"
            )
        
        self.vocab.clear()
        self.inverse_vocab.clear()
        self.bpe_merges.clear()
        self.merge_ranks.clear()

        unique_chars = self._collect_unique_chars(file_path, end_token)
        self._initialize_vocab(unique_chars, special_tokens)
        del unique_chars

        words = self._build_training_words(
            file_path, end_token, sample_size, seed
        )
        if not words:
            raise ValueError(
                "No protein sequences were found in the corpus."
            )

        pair_counts, pair_to_words = self._build_pair_statistics(words)

        next_id = len(self.vocab)

        with tqdm(total=vocab_size-next_id, desc="Training BPE", unit="merge") as pbar:
            while next_id < vocab_size:
                if not pair_counts:
                    break
                best_packed, best_count = pair_counts.most_common(1)[0]

                if best_count <= 0:
                    break

                best_a, best_b = _unpack(best_packed)
                new_id = next_id
                self.bpe_merges[(best_a, best_b)] = new_id
                self.merge_ranks[(best_a, best_b)] = len(self.merge_ranks)

                affected = pair_to_words.pop(best_packed, set())
                pair_counts.pop(best_packed, None)

                for idx in affected:
                    w = words[idx]

                    for i in range(len(w) - 1):
                        pair = _pack(w[i], w[i+1])
                        pair_counts[pair] -= 1
                        if pair_counts[pair] <= 0:
                            pair_counts.pop(pair, None)
                        pair_to_words[pair].discard(idx)
                    

                    merged = self._merge_word(w, best_a, best_b, new_id)
                    words[idx] = merged

                    for i in range(len(merged)-1):
                        pair = _pack(merged[i], merged[i+1])
                        pair_counts[pair] += 1
                        pair_to_words[pair].add(idx)
                next_id += 1
                pbar.update(1)

        for (a, b), new_id in self.bpe_merges.items():
            token_a = self.vocab[a]
            token_b = self.vocab[b]

            merged_token = (
                token_a, token_b
            )
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

        del words
        del pair_counts
        del pair_to_words
    
    def encode(self, text, allowed_special=None):
        if allowed_special is None:
            allowed_special = set()

        special_tokens = sorted(
            allowed_special,
            key=len,
            reverse=True
        )

        token_ids = []
        i = 0
        while i < len(text):
            matched = False
            for special_token in special_tokens:
                if text.startswith(special_token, i):
                    token_id = self.inverse_vocab.get(special_token)
                    if token_id is None:
                        raise ValueError(f"Special token '{special_tokens}' not found in vocabulary.")

                    token_ids.append(token_id)
                    i+=len(special_token)
                    matched = True
                    break
            if matched:
                continue
            j = i

            while j < len(text):
                if any(text.startswith(special_token, j)
                       for special_token in special_tokens):
                    break
                j+=1
            chunk = text[i:j]
            chunk_ids = self.tokenize(chunk)
            token_ids.extend(chunk_ids)

            i = j
        return token_ids


    def decode(self, token_ids):
        def decode_token(token):
            if isinstance(token, tuple):
                return "".join(decode_token(part) for part in token)
            return token
        
        decoded = []
        for token_id in token_ids:
            token = self.vocab.get(token_id)
            if token is None:
                raise ValueError(
                    f"Token ID {token_id} not found in vocabulary."
                )
            decoded.append(decode_token(token))
        return "".join(decoded)

    def tokenize(self, text):
        unk_id = self.inverse_vocab[self.unk_token]

        token_ids = [self.inverse_vocab.get(char, unk_id) 
                     for char in text]
        if len(token_ids) < 2:
            return token_ids

        while len(token_ids) >= 2:
            best_index = None
            best_rank = None

            for i in range(len(token_ids) - 1):
                pair = (token_ids[i], token_ids[i+1])
                rank = self.merge_ranks.get(pair)

                if rank is None:
                    continue
                if best_rank is None or rank < best_rank:
                    best_rank = rank
                    best_index = i
            if best_index is None:
                break
            pair = (
                token_ids[best_index],
                token_ids[best_index+1]
            )
            new_id = self.bpe_merges[pair]
            token_ids[best_index: best_index+2] = [new_id]
        return token_ids
    
    def save_vocab_and_merges(self, vocab_path, merges_path):
        with open(vocab_path, 'w', encoding='utf-8') as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        merges_list = []
        for rank, ((a,b), new_id) in enumerate(self.bpe_merges.items()):
            merges_list.append(
                {
                    'pair': [a, b],
                    'new_id': new_id,
                    'rank': rank
                }
            )

        with open(merges_path, 'w', encoding='utf-8') as file:
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, merges_path):
        self.vocab.clear()
        self.inverse_vocab.clear()
        self.bpe_merges.clear()
        self.merge_ranks.clear()

        with open(vocab_path, 'r', encoding='utf-8') as file:
            loaded_vocab = json.load(file)

            self.vocab = {
                int(k): self._convert_to_tuple(v)
                for k, v in loaded_vocab.items()
            }

            self.inverse_vocab = {
                v: k
                for k, v in self.vocab.items()
            }

        with open(merges_path, 'r', encoding='utf-8') as file:
            merges_list = json.load(file)

            for default_rank, merge in enumerate(merges_list):
                pair = tuple(merge['pair'])
                new_id = merge['new_id']
                rank = merge.get("rank", default_rank)

                self.bpe_merges[pair] = new_id
                self.merge_ranks[pair] = rank

        return self

In [7]:
class Normalization(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x-mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [8]:
class GELU(torch.nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [9]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4*emb_dim),
            GELU(),
            nn.Linear(4*emb_dim, emb_dim)
        )
    
    def forward(self, x):
        return self.layers(x)

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
        dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by number of heads."

        self.d_out = d_out
        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads

        self.Wquery = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Wkey = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Wvalue = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1))
    
    def forward(self, x): # x: position embedding + token embedding
        b, num_tokens, d_in = x.shape
        
        queries = self.Wquery(x) 
        keys = self.Wkey(x)
        values = self.Wvalue(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # b, num_heads, num_tokens, head_dim
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        atten_scores = queries @ keys.transpose(2, 3)

        atten_scores.masked_fill_(self.mask[:num_tokens, :num_tokens]
                                  , -torch.inf)

        atten_weight = torch.softmax(atten_scores / keys.shape[-1]**0.5, dim=-1)
        atten_weight = self.dropout(atten_weight)

        context_vec = (atten_weight @ values).transpose(1, 2)
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.proj(context_vec)
        return context_vec

In [11]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, d_out, context_length, dropout, num_heads, dkv_bias):
        super().__init__()
        self.norm1 = Normalization(emb_dim)
        self.attention = MultiHeadAttention(d_in=emb_dim, d_out=d_out,
         context_length=context_length, dropout=dropout,
          num_heads=num_heads , qkv_bias=dkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.norm2 = Normalization(emb_dim)
        self.ff = FeedForward(emb_dim)
    
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = x + shortcut
        return x

In [12]:
class GPT2(nn.Module):
    def __init__(self, emb_dim, d_out, vocab_size,
                  context_length, num_heads,
                    n_layers, dropout, qkv_bias):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(context_length, emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(emb_dim=emb_dim, d_out=d_out,
             context_length=context_length, dropout=dropout,
              num_heads=num_heads, dkv_bias=qkv_bias) 
            for _ in range(n_layers)]
        )
        self.norm = Normalization(emb_dim)
        self.out_head = nn.Linear(emb_dim, vocab_size, bias=False)

    def forward(self, in_idx):
        b, seq_len = in_idx.shape
        positions = torch.arange(seq_len, device=in_idx.device)
        tok_embedds = self.tok_emb(in_idx)
        pos_embedds = self.pos_emb(positions)
        x = tok_embedds + pos_embedds
        x = self.dropout(x)
        x = self.transformer_blocks(x)
        x = self.norm(x)
        logits = self.out_head(x)
        return logits

In [13]:
def calculate_loss_batch(input_batch, target_batch, model, device, unk_id=None):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = F.cross_entropy(logits.flatten(0, 1), target_batch.flatten(), ignore_index=unk_id if unk_id is not None else -100)
    return loss

def calculate_loss_loader(data_loader, model, device, unk_id=None, num_batches=None):
    total_loss = 0.0
    batch_count = 0

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if num_batches is not None and i >= num_batches:
            break
        loss = calculate_loss_batch(input_batch, target_batch,
                                     model, device, unk_id)
        total_loss += loss.item()
        batch_count += 1

    if batch_count == 0:
        return float('nan')

    return total_loss / batch_count


In [14]:
def evaluate_model(model, train_loader, val_loader,
                    device, eval_iter, unk_id=None):
    model.eval()
    with torch.no_grad():
        train_loss = calculate_loss_loader(train_loader, model, device, unk_id, eval_iter)
        val_loss = calculate_loss_loader(val_loader, model, device, unk_id, eval_iter)
    model.train()
    return train_loss, val_loss

In [15]:
def save_model(checkpoint, checkpoint_path, save_file_name):
    Path(checkpoint_path).mkdir(parents=True, exist_ok=True)
    checkpoint_path = os.path.join(checkpoint_path, save_file_name)
    torch.save(checkpoint, checkpoint_path)

In [16]:
def train_model(model, train_loader, val_loader, train_val_loader, optimizer, scheduler,
                 device, num_epochs, eval_freq, eval_iter, train_position_array, train_idx_array,
                   num_workers, checkpoint_path='ml/src/training/checkpoint', save_file_name=None, unk_id=None):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, steps = 0, -1
    start_epoch = 0

    checkpoint_file = os.path.join(
        checkpoint_path,
        save_file_name
    )
    if os.path.exists(checkpoint_file):
        checkpoint = torch.load(
            checkpoint_file,
            map_location='cpu',
            weights_only=False
        )

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        for state in optimizer.state.values():
            for k, v in state.items():
                if torch.is_tensor(v):
                    state[k] = v.to(device)
                
        start_epoch = checkpoint["epoch"]
        steps = checkpoint["steps"]
        tokens_seen = checkpoint["tokens_seen"]

        del checkpoint
    model.to(device)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    for epoch in range(start_epoch, num_epochs, 1):
        model.train()
        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}",
            unit="batch"
        )
        for input_batch, target_batch in progress_bar:
            optimizer.zero_grad()
            loss = calculate_loss_batch(input_batch, target_batch, model, device, unk_id)
            loss.backward()
            optimizer.step()
            scheduler.step()
            tokens_seen += input_batch.numel()
            steps += 1

            progress_bar.set_postfix(loss=f"{loss.item():.3f}")

            if steps % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_val_loader,
                                                      val_loader, device, eval_iter, unk_id)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch {epoch+1} (Step {steps}):")
                print(f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")
                print(f"Train perplexity {math.exp(train_loss):.3f}, Val perplexity {math.exp(val_loss):.3f}")

                if save_file_name is not None:
                    worker_byte_offsets = {i: train_position_array[i] for i in range(len(train_position_array))}
                    worker_next_idx     = {i: train_idx_array[i] for i in range(len(train_idx_array))}
                    checkpoint = {
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "epoch": epoch,
                        "steps": steps,
                        "tokens_seen": tokens_seen,
                        "worker_byte_offsets": worker_byte_offsets,
                        "worker_next_idx": worker_next_idx,
                        "num_workers": num_workers,
                    }
                    save_model(checkpoint, checkpoint_path, save_file_name)
    
    return train_losses, val_losses, track_tokens_seen

In [17]:
vocab_path = '/kaggle/input/datasets/mersadmostofian0/proteinas-dataset/proteinas_bpe_vocab.json'
merges_path = '/kaggle/input/datasets/mersadmostofian0/proteinas-dataset/proteinas_bpe_merges.json'
train_corpus_path = '/kaggle/input/datasets/mersadmostofian0/proteinas-dataset/proteinas_train.txt'
val_corpus_path = '/kaggle/input/datasets/mersadmostofian0/proteinas-dataset/proteinas_test.txt'
end_token='<|endofprotein|>'
checkpoint_path= './checkpoints/'
save_file_name='checkpoint-model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = BPETokenizer().load_vocab_and_merges(vocab_path, merges_path)


In [18]:
config = {
    'vocab_size':5000,
    'context_length':512,
    'emb_dim':512,
    'n_heads':8,
    'n_layers':8,
    'drop_rate':0.1,
    'qkv_bias':True,
}

In [19]:
torch.manual_seed(123)


In [20]:
num_workers = 1
train_position_array = mp.Array('q', num_workers)
train_idx_array = mp.Array('q', num_workers)

resume_state = {}
if os.path.exists(os.path.join(checkpoint_path, save_file_name)):
    checkpoint = torch.load(os.path.join(checkpoint_path, save_file_name), map_location='cpu', weights_only=False)
    saved_offset = checkpoint.get('worker_byte_offsets', {})
    saved_idxs = checkpoint.get('worker_next_idx', {})
    resume_state = {
        w: (saved_offset.get(w, 0), saved_idxs.get(w, 0))
        for w in range(num_workers)
    }

In [21]:
train_data = ProteinDataset(
    corpus_path=train_corpus_path, tokenizer=tokenizer,
    context_length=config['context_length'], end_token=end_token,
    resume_state=resume_state, position_array=train_position_array,
    idx_array=train_idx_array
)
train_eval_dataset = ProteinDataset(
    corpus_path=train_corpus_path,
    tokenizer=tokenizer,
    context_length=config['context_length'],
    end_token="<|endofprotein|>",
    buffer_size=1,
)
val_data = ProteinDataset(corpus_path=val_corpus_path, tokenizer=tokenizer,
                         context_length=config['context_length'], end_token=end_token)

In [22]:
train_loader = DataLoader(
    train_data,
    batch_size=8,
    num_workers=num_workers,
    pin_memory=True,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=2,
    num_workers=0,
)

val_loader = DataLoader(
    val_data,
    batch_size=2,
    num_workers=num_workers,
    pin_memory=True,
)

In [23]:
gpt2_model = GPT2(
    emb_dim=config['emb_dim'],
    d_out=config['emb_dim'],
    vocab_size=config['vocab_size'],
    context_length=config['context_length'],
    num_heads=config['n_heads'],
    n_layers=config['n_layers'],
    dropout=config['drop_rate'],
    qkv_bias=config['qkv_bias']
)

In [24]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
optimizer = torch.optim.AdamW(gpt2_model.parameters(), lr=3e-4, weight_decay=0.1)

warmup_steps = 500
total_steps = 500_000

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=warmup_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-5
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)


In [25]:
train_losses, val_losses, track_tokens_seen = train_model(gpt2_model, train_loader, val_loader, train_eval_loader,
             optimizer, scheduler, device,15, 256, 200, train_position_array, train_idx_array, num_workers, checkpoint_path, save_file_name)

Epoch 1/15: 0batch [00:08, ?batch/s, loss=8.694]

Epoch 1 (Step 0):
Train loss 8.686, Val loss 8.685
Train perplexity 5920.235, Val perplexity 5915.442


Epoch 1/15: 256batch [01:28,  4.48batch/s, loss=7.099]

Epoch 1 (Step 256):
Train loss 7.126, Val loss 7.129
Train perplexity 1244.201, Val perplexity 1247.022


Epoch 1/15: 303batch [02:03,  2.45batch/s, loss=7.140]
Epoch 2/15: 209batch [00:53,  4.49batch/s, loss=7.111]

...

Epoch 13/15: 303batch [01:39,  3.04batch/s, loss=5.511]
Epoch 14/15: 157batch [00:42,  4.49batch/s, loss=5.203]

Epoch 14 (Step 4096):
Train loss 4.004, Val loss 7.915
Train perplexity 54.827, Val perplexity 2737.181


Epoch 14/15: 303batch [01:39,  3.04batch/s, loss=5.225]
Epoch 15/15: 110batch [00:31,  4.49batch/s, loss=4.847]

Epoch 15 (Step 4352):
Train loss 3.410, Val loss 8.164
Train perplexity 30.262, Val perplexity 3513.028


Epoch 15/15: 303batch [01:39,  3.05batch/s, loss=4.976]


In [26]:
torch.save(gpt2_model.state_dict(), 'protbuildv1.pth')


In [27]:
def generate_sequnce(
        model,
        idx,
        max_new_tokens,
        context_size,
        end_token_id=None,
        temperature=0,
        topk=None
):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if topk is not None:
            top_logits, _ = torch.topk(logits, topk)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(logits.device), logits)

        if temperature > 0:
            logits = logits / temperature
            logits = logits - logits.max(dim=-1, keepdim=True).values
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, 1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next), dim=1)
        if end_token_id is not None:
            if torch.all(idx_next == end_token_id):
                break
    return idx

def generate_protein(model, prompt, tokenizer,
                      max_new_tokens, context_size,
                        device, end_token="<|endofprotein|>",
                        temperature=0, topk=None):
    token_ids = tokenizer.encode(prompt, allowed_special={end_token})
    idx = torch.tensor(
        token_ids,
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    end_token_id = tokenizer.inverse_vocab[end_token]
    idx = generate_sequnce(model, idx, max_new_tokens,
                            context_size, end_token_id=end_token_id,
                            temperature=temperature, topk=topk)

    return tokenizer.decode(idx[0].tolist())

In [28]:
prompts = [
    "MKWVTFISLLLLFSSAYSRGVFRR",
    "MKTIIALSYIFCLVFAD",
    "MVLSPADKTNVKAAWGKVGA"
]

for prompt in prompts:
    print("=" * 80)
    print(prompt)

    output = generate_protein(
        gpt2_model,
        prompt=prompt,
        tokenizer=tokenizer,
        max_new_tokens=100,
        context_size=config["context_length"],
        device=device,
        temperature=1.0,
        topk=25
    )

    print(output)

MKWVTFISLLLLFSSAYSRGVFRR
MKWVTFISLLLLFSSAYSRGVFRRPVPRVPLHNYNTRDWSSMCMPVDANKDYPATWSMTCITNLYCIQQWHEDFLSTRNVIMNVIGYMDCTLTHKRHRKVNRVCSYMGKHTWDVWRIANRRRLSEMAWLWNRTNMSATNFT<|endofprotein|>
MKTIIALSYIFCLVFAD
MKTIIALSYIFCLVFADEFECAHAPKCMFTFRGLFRQDLQWRRCLM<|endofprotein|>
MVLSPADKTNVKAAWGKVGA
MVLSPADKTNVKAAWGKVGALVQYQTTLKWWVFCRWRMKQPLERPIGGYKYDGVSNRPC<|endofprotein|>


In [29]:
temperatures = [0, 0.5, 0.8, 1.0, 1.2]

for temperature in temperatures:
    output = generate_protein(
        gpt2_model,
        prompt="MVLSPADKTNVKAAWGKVGA",
        tokenizer=tokenizer,
        max_new_tokens=100,
        context_size=config["context_length"],
        device=device,
        temperature=temperature,
        topk=25
    )

    print(f"\nTemperature = {temperature}")
    print(output)


Temperature = 0
MVLSPADKTNVKAAWGKVGAWDWWLLHVFQNVFQKIMVCWNPPFHVPQYYYGSEYMQMAQHVFAEGWPTL<|endofprotein|>

Temperature = 0.5
MVLSPADKTNVKAAWGKVGAQNEWQNTMSQEQHYNIWHQMPQGCW<|endofprotein|>

Temperature = 0.8
MVLSPADKTNVKAAWGKVGAMFWLEIQKHGRIWIMNECTTRCCHGETSMNLYMKEQGNCWPYGIVFVYS<|endofprotein|>

Temperature = 1.0
MVLSPADKTNVKAAWGKVGAVCPNFCRIPFRYRMATHFPMETMSRQKQNRNEHRFCFGMSTTHWNMGQTCPVFFLIQIPYWIC<|endofprotein|>

Temperature = 1.2
MVLSPADKTNVKAAWGKVGARNWIVKRAHYFVFEYCFLHNNGNYEPEIVFPKFRGEGCKSYVAHCHQEHMLWYEIQITA<|endofprotein|>
